In [ ]:
!pip install numpy 
!pip install numba

In [ ]:
import numpy as np
from numba import vectorize, float64
import time

K = 21
ITER = 5

@vectorize([float64(float64, float64)], target='cuda', nopython=True)
def approx_pi(i, num_elem):
    nn = num_elem * num_elem
    return nn / (nn + i * i - i + 0.25)

if __name__=="__main__":
    num_elem = 1 << K

    measurements = []

    for _ in range(ITER):
        time_start = time.perf_counter_ns()

        pi_approx = np.sum(approx_pi(np.arange(num_elem, dtype=np.float64), num_elem)) * 4.0 / num_elem

        time_end = time.perf_counter_ns()

        print(f"Approx: {pi_approx}")

        measurements.append(f"{K} {(time_end - time_start) // 1000}")

    print("Measurements:")
    for measurement in measurements:
        print(measurement)

In [ ]:
import numpy as np
from numba import guvectorize, float64
import time

K = 21
ITER = 5
BLOCK_SIZES = [1024, 2048, 4096, 8192, 16384]

@guvectorize([(float64[:], float64, float64[:])], '(n),()->()', target='cuda', nopython=True)
def approx_pi(block, num_elem, res):
    sum = 0.0
    nn = num_elem * num_elem
    
    for i in range(block.shape[0]):
        x = block[i]
        sum += nn / (nn + x * x - x + 0.25)
    res[0] = sum

if __name__=="__main__":
    num_elem = 1 << K

    measurements = []

    for block_size in BLOCK_SIZES:
        num_blocks = num_elem // block_size

        for _ in range(ITER):
            time_start = time.perf_counter_ns()

            ins = np.arange(num_elem, dtype=np.float64)
            ins = ins.reshape((num_blocks, block_size))

            results = approx_pi(ins, np.float64(num_elem))

            pi_approx = np.sum(results) * 4.0 / num_elem

            time_end = time.perf_counter_ns()

            print(f"Approx: {pi_approx}")

            measurements.append(f"{K} {block_size} {(time_end - time_start) // 1000}")

    print("Measurements:")
    for measurement in measurements:
        print(measurement)